In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [2]:
# Select modules to keep:
modules_to_keepL = np.array([9, 14]) # And the modules + 24
channels_to_keepL = np.array([n * 128 + m for n in modules_to_keepL for m in range(128)])
modules_to_keepR = np.array([10, 13])
channels_to_keepR = np.array([n * 128 + m for n in modules_to_keepR for m in range(128)])
# Data
data_dir = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hori.bin'
#num_rows = os.path.getsize(data_dir) // 6
num_rows = os.path.getsize(data_dir) // 8
#data = np.memmap(data_dir, dtype = np.int16, shape=(num_rows, 3))
data = np.memmap(data_dir, dtype = np.int16, shape=(num_rows, 4))
print(num_rows)
print(data[0:10])

44764206
[[ 256 1188 1075    0]
 [ 388 1210  437    0]
 [1762 1812 1867    0]
 [ 584 1902 3302    0]
 [1715 1911 1252    0]
 [ 286 1827  610    0]
 [2034 1322 1290    0]
 [ 768 1180 1395    0]
 [1516 1172 7290    0]
 [1457 1903  967    0]]


In [41]:
output_dir = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hori_geomasked.bin'
for i in tqdm(range(0, num_rows, 1000000)):
    chunk = np.copy(data[i:i+1000000])
    # Create a set for faster lookup

    # Create masks for chunk[:,0] and chunk[:,1] to be in channels_to_keep
    mask0 = np.isin(chunk[:, 0], channels_to_keepL)
    mask1 = np.isin(chunk[:, 1], channels_to_keepR)

    # Keep only rows where both chunk[:,0] and chunk[:,1] are in channels_to_keep
    mask = mask0 & mask1
    chunk = chunk[mask]

    with open(output_dir, 'ab') as f:
        chunk.tofile(f)

100%|██████████| 45/45 [00:00<00:00, 53.43it/s]


In [10]:
print(channels_to_keep_L)

[4224 4225 4226 4227 4228 4229 4230 4231 4232 4233 4234 4235 4236 4237
 4238 4239 4240 4241 4242 4243 4244 4245 4246 4247 4248 4249 4250 4251
 4252 4253 4254 4255 4256 4257 4258 4259 4260 4261 4262 4263 4264 4265
 4266 4267 4268 4269 4270 4271 4272 4273 4274 4275 4276 4277 4278 4279
 4280 4281 4282 4283 4284 4285 4286 4287 4288 4289 4290 4291 4292 4293
 4294 4295 4296 4297 4298 4299 4300 4301 4302 4303 4304 4305 4306 4307
 4308 4309 4310 4311 4312 4313 4314 4315 4316 4317 4318 4319 4320 4321
 4322 4323 4324 4325 4326 4327 4328 4329 4330 4331 4332 4333 4334 4335
 4336 4337 4338 4339 4340 4341 4342 4343 4344 4345 4346 4347 4348 4349
 4350 4351 4352 4353 4354 4355 4356 4357 4358 4359 4360 4361 4362 4363
 4364 4365 4366 4367 4368 4369 4370 4371 4372 4373 4374 4375 4376 4377
 4378 4379 4380 4381 4382 4383 4384 4385 4386 4387 4388 4389 4390 4391
 4392 4393 4394 4395 4396 4397 4398 4399 4400 4401 4402 4403 4404 4405
 4406 4407 4408 4409 4410 4411 4412 4413 4414 4415 4416 4417 4418 4419
 4420 

In [ ]:
# Create a 3D plot of the scanner
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Plot the scanner lines
ax.scatter(map[:,0], map[:,1], map[:,2], marker='o', color = 'gray', alpha = 0.5, s=1)

# Plot a single range
modulenum = 38
ax.scatter(map[:,0][modulenum * 128: (modulenum + 1) * 128], map[:,1][modulenum * 128: (modulenum + 1) * 128], map[:,2][modulenum * 128: (modulenum + 1) * 128], marker='o', color = 'red', alpha = 1, s=1)
print(modulenum * 128, (modulenum + 1) * 128)

ax.set_xlim(-150, 150)
ax.set_ylim(-150, 150)
ax.set_zlim(-100, 100)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
# Rotate the 3D view for better visualization (optional example: elev=30, azim=60)
ax.view_init(elev=30, azim=45)

plt.show()

In [44]:
# Cut on specific time
data_dir = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hori_geomasked.bin'
num_rows = os.path.getsize(data_dir) // 8
data = np.memmap(data_dir, dtype = np.int16, shape=(num_rows, 4))
out_dir = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hori_geomasked_timecut.bin'
#print(data[0:5])
#print(np.max(data[:, 3]))

endtime = 2346
linelength = 280
def get_timerange(radius):
    time1 = radius / (linelength / 2) * endtime + endtime / 2
    time2 = 2346 - time1
    return time1, time2

if os.path.exists(out_dir):
    os.remove(out_dir)
for i in tqdm(range(0, num_rows, 1000000)):
    chunk = np.copy(data[i:i+1000000])
    time1, time2 = get_timerange(10)
    time1 = np.int16(time1)
    time2 = np.int16(time2)
    section1 = chunk[chunk[:, 3] > time1 - 4]
    section1 = section1[section1[:, 3] < time1 + 4]
    section2 = chunk[chunk[:, 3] > time2 - 4]
    section2 = section2[section2[:, 3] < time2 + 4]
    towrite = np.concatenate((section1, section2), axis = 0)
    with open(out_dir, 'ab') as f:
        towrite.tofile(f)



100%|██████████| 2/2 [00:00<00:00, 127.67it/s]


In [22]:
# Cut for non-oblique LORs


data_dir = '/home/kale-chen/Documents/PET/Spatial Resolution/Data/1723.bin'
num_rows = os.path.getsize(data_dir) // 12
data = np.fromfile(data_dir, dtype=np.int16).reshape(num_rows, 6)
print(len(data), data[0:5])

mapdf = pd.read_csv('/home/kale-chen/Documents/PET/TPPT_Scanner_map_adjusted.csv', usecols=[0,1,2,3,4,5], header=None)
map = mapdf.to_numpy()

zLs = map[data[:,1] + 3072, 2]
zRs = map[data[:,3], 2]
print(len(zLs), len(zRs))
mask = np.abs(zLs - zRs) < 10
print(np.sum(mask), len(mask))
print(mask[0:20])
data = data[mask]
print(data[0:5])
print(len(data))



7759517 [[2654  489 2003  835  716    1]
 [1874 1281 2051 1798  238    1]
 [1974 2196 2033 2367 -228    1]
 [1872 2223 1982 2294  191    1]
 [2218 1303 2273 1181  309    1]]
7759517 7759517
1444317 7759517
[False False False False  True False False False False False  True False
 False False False False False False  True False]
[[2218 1303 2273 1181  309    1]
 [2175 3016 2067 2766  506    1]
 [2344  793 2005  659  -52    1]
 [2155  210 2133  345  396    1]
 [2169 1867 2185 1728   75    1]]
1444317
